# 🎬 Automatic Video Highlight Generator
### Using CLIP-based Scene Scoring + Temporal Segmentation

**Pipeline:**
1. Download TVSum dataset directly
2. Extract frames from a video
3. Score each frame using CLIP against highlight-related prompts
4. Pick top-scoring segments
5. Stitch into a highlight clip using MoviePy
6. Summarize with BART (optional)

> ✅ No paid APIs. All free models. Runs fully on Kaggle GPU.

## Step 1 — Install Dependencies

In [1]:
%%capture
!pip install transformers moviepy ftfy regex tqdm Pillow torch torchvision --quiet
print('✅ All dependencies installed.')

## Step 2 — Download TVSum Dataset

In [6]:
import os

print("📥 Downloading TVSum dataset...")

# Remove old/corrupted files
!rm -f tvsum50_ver_1_1.tgz

# Download
!wget -c http://people.csail.mit.edu/yalesong/tvsum/tvsum50_ver_1_1.tgz

print("✅ Download complete")

# Extract
!tar -xvzf tvsum50_ver_1_1.tgz

print("✅ Extraction complete")

# Check contents
!ls

📥 Downloading TVSum dataset...
--2026-05-20 11:57:48--  http://people.csail.mit.edu/yalesong/tvsum/tvsum50_ver_1_1.tgz
Resolving people.csail.mit.edu (people.csail.mit.edu)... 128.52.131.233
Connecting to people.csail.mit.edu (people.csail.mit.edu)|128.52.131.233|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://people.csail.mit.edu/yalesong/tvsum/tvsum50_ver_1_1.tgz [following]
--2026-05-20 11:57:48--  https://people.csail.mit.edu/yalesong/tvsum/tvsum50_ver_1_1.tgz
Connecting to people.csail.mit.edu (people.csail.mit.edu)|128.52.131.233|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 671779858 (641M) [application/x-gzip]
Saving to: ‘tvsum50_ver_1_1.tgz’

tvsum50_ver_1_1.tgz 100%[===================>] 640.66M  10.5MB/s    in 60s     

2026-05-20 11:58:48 (10.7 MB/s) - ‘tvsum50_ver_1_1.tgz’ saved [671779858/671779858]

✅ Download complete
./WebscopeReadMe.txt
./ydata-tvsum50-v1_1/
./ydata-tvsum50-v1_1/README
./yda

In [8]:
!ls ydata-tvsum50-v1_1

README			ydata-tvsum50-matlab.zip     ydata-tvsum50-video.zip
ydata-tvsum50-data.zip	ydata-tvsum50-thumbnail.zip


In [9]:
# Extract annotation data
!unzip -q ydata-tvsum50-v1_1/ydata-tvsum50-data.zip -d tvsum_data

In [10]:
# Extract videos
!unzip -q ydata-tvsum50-v1_1/ydata-tvsum50-video.zip -d tvsum_videos

In [11]:
# Extract thumbnails
!unzip -q ydata-tvsum50-v1_1/ydata-tvsum50-thumbnail.zip -d tvsum_thumbnails

In [12]:
# Extract matlab files
!unzip -q ydata-tvsum50-v1_1/ydata-tvsum50-matlab.zip -d tvsum_matlab

In [13]:
!ls


tvsum50_ver_1_1.tgz  tvsum_matlab      tvsum_videos	   ydata-tvsum50-v1_1
tvsum_data	     tvsum_thumbnails  WebscopeReadMe.txt


In [15]:
!ls  tvsum_videos/video

0tmA_C6XwfM.mp4  EE-bNr36nyA.mp4  JgHubY5Vw3Y.mp4  VuWGsYPqAX8.mp4
37rzWOQsNIw.mp4  eQu1rNs0an0.mp4  JKpqYvAdIsw.mp4  WG0MBPpPC6I.mp4
3eYKfiOEJNs.mp4  -esJrBWj2d8.mp4  kLxoNp-UchI.mp4  WxtbjNsCQ8A.mp4
4wU_LUjG5Ic.mp4  EYqVtI9YWJA.mp4  LRw_obCPUt0.mp4  XkqCExn6_Us.mp4
91IHQYk1IQM.mp4  fWutDQy1nnY.mp4  NyBmCxDoHJU.mp4  xmEERLqJ2kU.mp4
98MoyGZKHXc.mp4  GsAD1KT1xo8.mp4  oDXZc0tZe04.mp4  _xMr-HKMfVA.mp4
akI8YFjEmUw.mp4  gzDbaEs1Rlg.mp4  PJrm840pAUI.mp4  xwqBXPGE9pQ.mp4
AwmHb44_ouw.mp4  Hl-__g2gn_A.mp4  qqR6AEXwxoQ.mp4  xxdtq8mxegs.mp4
b626MiF1ew4.mp4  HT5vyqe0Xaw.mp4  RBCABdttQmI.mp4  XzYM3PfTM4w.mp4
Bhxk-O1Y7Ho.mp4  i3wAGJaaktw.mp4  Se3oxnaPsz0.mp4  Yi4Ij2NM7U4.mp4
byxOvuiIJV0.mp4  iVt07TCkFM0.mp4  sTEELN-vY30.mp4  z_6gVvQb2d0.mp4
cjibtmSLxQ4.mp4  J0nA4VgnoCo.mp4  uGu_10sucQo.mp4
E11zDS9XGzg.mp4  jcoYJXDG9sw.mp4  vdmoEJ5YbrQ.mp4


In [17]:
!ls tvsum_matlab/matlab

knapsack  script_evaluate_result.m  solve_knapsack.m  ydata-tvsum50.mat


In [18]:
!ls tvsum_data/data

ydata-tvsum50-anno.tsv	ydata-tvsum50-info.tsv


In [21]:
!ls tvsum_thumbnails/thumbnail


0tmA_C6XwfM.jpg  EE-bNr36nyA.jpg  JgHubY5Vw3Y.jpg  VuWGsYPqAX8.jpg
37rzWOQsNIw.jpg  eQu1rNs0an0.jpg  JKpqYvAdIsw.jpg  WG0MBPpPC6I.jpg
3eYKfiOEJNs.jpg  -esJrBWj2d8.jpg  kLxoNp-UchI.jpg  WxtbjNsCQ8A.jpg
4wU_LUjG5Ic.jpg  EYqVtI9YWJA.jpg  LRw_obCPUt0.jpg  XkqCExn6_Us.jpg
91IHQYk1IQM.jpg  fWutDQy1nnY.jpg  NyBmCxDoHJU.jpg  xmEERLqJ2kU.jpg
98MoyGZKHXc.jpg  GsAD1KT1xo8.jpg  oDXZc0tZe04.jpg  _xMr-HKMfVA.jpg
akI8YFjEmUw.jpg  gzDbaEs1Rlg.jpg  PJrm840pAUI.jpg  xwqBXPGE9pQ.jpg
AwmHb44_ouw.jpg  Hl-__g2gn_A.jpg  qqR6AEXwxoQ.jpg  xxdtq8mxegs.jpg
b626MiF1ew4.jpg  HT5vyqe0Xaw.jpg  RBCABdttQmI.jpg  XzYM3PfTM4w.jpg
Bhxk-O1Y7Ho.jpg  i3wAGJaaktw.jpg  Se3oxnaPsz0.jpg  Yi4Ij2NM7U4.jpg
byxOvuiIJV0.jpg  iVt07TCkFM0.jpg  sTEELN-vY30.jpg  z_6gVvQb2d0.jpg
cjibtmSLxQ4.jpg  J0nA4VgnoCo.jpg  uGu_10sucQo.jpg
E11zDS9XGzg.jpg  jcoYJXDG9sw.jpg  vdmoEJ5YbrQ.jpg


## Step 3 — Download a Sample Video from TVSum

TVSum contains 50 YouTube video IDs. We pick one and download it with `yt-dlp` (free, no API key needed).

In [22]:
import json
import scipy.io
import numpy as np

%%capture
!pip install yt-dlp --quiet

# Load TVSum annotations to get video IDs
mat = scipy.io.loadmat('tvsum_matlab/matlab/ydata-tvsum50.mat', simplify_cells=True)
videos = mat['tvsum50']['video']
print(f'Total videos in TVSum: {len(videos)}')

# Pick the first video ID
video_id = videos[0]['name']  # YouTube video ID
gt_scores = np.array(videos[0]['user_anno'], dtype=float).mean(axis=0)  # Ground truth importance
print(f'\n🎬 Chosen Video ID: {video_id}')
print(f'Ground truth score frames: {len(gt_scores)}')

UsageError: Line magic function `%%capture` not found.


In [ ]:
video_path = f'{video_id}.mp4'

if not os.path.exists(video_path):
    print(f'📥 Downloading video {video_id} from YouTube...')
    !yt-dlp -f "bestvideo[ext=mp4][height<=480]+bestaudio[ext=m4a]/best[ext=mp4][height<=480]" \
        --merge-output-format mp4 \
        -o "{video_path}" \
        "https://www.youtube.com/watch?v={video_id}"
else:
    print(f'✅ Video already exists: {video_path}')

print('\n✅ Video ready.')

## Step 4 — Extract Frames from Video

In [ ]:
import cv2
from PIL import Image

FRAME_SAMPLE_RATE = 2  # Extract 1 frame every 2 seconds
FRAMES_DIR = 'extracted_frames'
os.makedirs(FRAMES_DIR, exist_ok=True)

def extract_frames(video_path, sample_rate=2):
    """Extract frames from video at given sample rate (seconds)."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps
    
    print(f'Video FPS     : {fps:.1f}')
    print(f'Total frames  : {total_frames}')
    print(f'Duration      : {duration:.1f}s ({duration/60:.1f} min)')
    
    frame_interval = int(fps * sample_rate)
    frames_data = []  # list of (frame_index, timestamp, pil_image)
    
    frame_idx = 0
    saved = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            timestamp = frame_idx / fps
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb)
            frames_data.append((frame_idx, timestamp, pil_img))
            saved += 1
        frame_idx += 1
    
    cap.release()
    print(f'\n✅ Extracted {saved} frames (every {sample_rate}s)')
    return frames_data, fps, duration

frames_data, fps, duration = extract_frames(video_path, FRAME_SAMPLE_RATE)

# Preview a few frames
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
step = max(1, len(frames_data) // 5)
for i, ax in enumerate(axes):
    idx = min(i * step, len(frames_data) - 1)
    fdata = frames_data[idx]
    ax.imshow(fdata[2])
    ax.set_title(f't={fdata[1]:.1f}s', fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Extracted Frames', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_frames.png', dpi=80, bbox_inches='tight')
plt.show()
print('✅ Frame preview saved.')

## Step 5 — Load CLIP Model (Free, HuggingFace)

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🔧 Using device: {DEVICE}')

print('📥 Loading CLIP ViT-B/32...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()
print('✅ CLIP loaded.')

## Step 6 — Score Frames Using CLIP

We score every frame against a set of **highlight-defining text prompts**. Higher similarity = more highlight-worthy.

In [ ]:
# --- HIGHLIGHT PROMPTS ---
# These define what a 'highlight' looks like to CLIP.
# Tune these based on video genre.
HIGHLIGHT_PROMPTS = [
    "an exciting and intense moment",
    "an emotional and dramatic scene",
    "a crowd cheering and celebrating",
    "an action-packed and thrilling moment",
    "a beautiful and cinematic shot",
    "a key moment in the story",
    "people reacting with strong emotion",
    "a climactic scene"
]

BORING_PROMPTS = [
    "a static and boring scene",
    "an empty room with nothing happening",
    "a dull and uneventful moment"
]

print(f'Scoring {len(frames_data)} frames against {len(HIGHLIGHT_PROMPTS)} highlight prompts...')

@torch.no_grad()
def score_frames_with_clip(frames_data, highlight_prompts, boring_prompts, batch_size=32):
    """Score each frame: avg highlight similarity minus avg boring similarity."""
    all_scores = []
    
    for i in range(0, len(frames_data), batch_size):
        batch = frames_data[i:i+batch_size]
        images = [f[2] for f in batch]
        
        # Encode images
        img_inputs = clip_processor(images=images, return_tensors='pt', padding=True).to(DEVICE)
        img_features = clip_model.get_image_features(**img_inputs)
        img_features = img_features / img_features.norm(dim=-1, keepdim=True)
        
        # Encode highlight prompts
        txt_inputs = clip_processor(text=highlight_prompts, return_tensors='pt', padding=True).to(DEVICE)
        txt_features = clip_model.get_text_features(**txt_inputs)
        txt_features = txt_features / txt_features.norm(dim=-1, keepdim=True)
        
        # Encode boring prompts
        bore_inputs = clip_processor(text=boring_prompts, return_tensors='pt', padding=True).to(DEVICE)
        bore_features = clip_model.get_text_features(**bore_inputs)
        bore_features = bore_features / bore_features.norm(dim=-1, keepdim=True)
        
        # Similarity scores
        highlight_sim = (img_features @ txt_features.T).mean(dim=-1)   # avg over prompts
        boring_sim    = (img_features @ bore_features.T).mean(dim=-1)
        
        # Final score = highlight - boring (contrast scoring)
        scores = (highlight_sim - boring_sim).cpu().numpy()
        all_scores.extend(scores.tolist())
        
        if (i // batch_size) % 5 == 0:
            print(f'  Processed {min(i+batch_size, len(frames_data))}/{len(frames_data)} frames...')
    
    return np.array(all_scores)

clip_scores = score_frames_with_clip(frames_data, HIGHLIGHT_PROMPTS, BORING_PROMPTS)

# Normalize scores to [0, 1]
clip_scores_norm = (clip_scores - clip_scores.min()) / (clip_scores.max() - clip_scores.min() + 1e-8)

print(f'\n✅ Scoring complete.')
print(f'Score range: {clip_scores_norm.min():.3f} → {clip_scores_norm.max():.3f}')
print(f'Mean score : {clip_scores_norm.mean():.3f}')

## Step 7 — Visualize CLIP Scores Over Time

In [ ]:
timestamps = np.array([f[1] for f in frames_data])

# Smooth scores for cleaner selection
from scipy.ndimage import uniform_filter1d
smoothed_scores = uniform_filter1d(clip_scores_norm, size=5)

# Threshold for highlight selection (top 20% of segments)
THRESHOLD = np.percentile(smoothed_scores, 80)

plt.figure(figsize=(16, 5))
plt.plot(timestamps, clip_scores_norm, alpha=0.4, color='steelblue', linewidth=1, label='Raw CLIP Score')
plt.plot(timestamps, smoothed_scores, color='navy', linewidth=2, label='Smoothed Score')
plt.axhline(y=THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Highlight Threshold (top 20%)')
plt.fill_between(timestamps, smoothed_scores, THRESHOLD,
                 where=(smoothed_scores >= THRESHOLD),
                 alpha=0.3, color='orange', label='Selected Highlights')
plt.xlabel('Time (seconds)', fontsize=12)
plt.ylabel('CLIP Highlight Score', fontsize=12)
plt.title('CLIP-based Scene Importance Scores Over Time', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('clip_scores_timeline.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Score timeline saved.')

## Step 8 — Select Highlight Segments

In [ ]:
HIGHLIGHT_DURATION_TARGET = 60   # Target total highlight length (seconds)
SEGMENT_PADDING = 2.0            # Seconds of padding around each selected frame
MIN_GAP_BETWEEN_SEGMENTS = 5.0   # Merge segments closer than this (seconds)

def select_highlight_segments(timestamps, scores, threshold, padding=2.0, min_gap=5.0):
    """Convert high-score frames into continuous time segments."""
    selected = [(timestamps[i], scores[i]) for i in range(len(scores)) if scores[i] >= threshold]
    
    if not selected:
        # Fallback: take top 20% by count
        n = max(1, len(scores) // 5)
        top_idx = np.argsort(scores)[-n:]
        selected = [(timestamps[i], scores[i]) for i in sorted(top_idx)]
    
    # Build raw segments with padding
    raw_segments = [(max(0, t - padding), t + padding) for t, _ in selected]
    
    # Merge overlapping / close segments
    merged = [raw_segments[0]]
    for start, end in raw_segments[1:]:
        if start - merged[-1][1] < min_gap:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    
    return merged

highlight_segments = select_highlight_segments(
    timestamps, smoothed_scores, THRESHOLD,
    padding=SEGMENT_PADDING,
    min_gap=MIN_GAP_BETWEEN_SEGMENTS
)

total_highlight_duration = sum(e - s for s, e in highlight_segments)

print(f'✅ Selected {len(highlight_segments)} highlight segments')
print(f'Total highlight duration: {total_highlight_duration:.1f}s ({total_highlight_duration/60:.1f} min)\n')
for i, (s, e) in enumerate(highlight_segments):
    print(f'  Segment {i+1:02d}: {s:.1f}s → {e:.1f}s  ({e-s:.1f}s)')

## Step 9 — Stitch Highlight Video with MoviePy

In [ ]:
from moviepy.editor import VideoFileClip, concatenate_videoclips
import warnings
warnings.filterwarnings('ignore')

OUTPUT_PATH = 'highlight_reel.mp4'
MAX_HIGHLIGHT_SECONDS = 90  # Cap total highlight at 90s

print('🎬 Stitching highlight reel...')
source_clip = VideoFileClip(video_path)
video_duration = source_clip.duration

clips = []
total_duration = 0

for start, end in highlight_segments:
    # Clamp to video duration
    start = max(0, min(start, video_duration - 0.1))
    end   = max(start + 0.5, min(end, video_duration))
    
    if total_duration + (end - start) > MAX_HIGHLIGHT_SECONDS:
        remaining = MAX_HIGHLIGHT_SECONDS - total_duration
        if remaining > 1.0:
            end = start + remaining
        else:
            break
    
    clip = source_clip.subclip(start, end)
    clips.append(clip)
    total_duration += (end - start)
    print(f'  ✂️  Added: {start:.1f}s → {end:.1f}s ({end-start:.1f}s)')

print(f'\nTotal highlight duration: {total_duration:.1f}s')

if clips:
    final_highlight = concatenate_videoclips(clips, method='compose')
    final_highlight.write_videofile(
        OUTPUT_PATH,
        codec='libx264',
        audio_codec='aac',
        verbose=False,
        logger=None
    )
    source_clip.close()
    print(f'\n✅ Highlight reel saved → {OUTPUT_PATH}')
    print(f'File size: {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB')
else:
    print('⚠️ No clips selected. Try lowering the threshold.')

## Step 10 — Optional: Generate Text Summary with BART

Describes what kind of content is in the highlight reel using the top CLIP-matched prompts.

In [ ]:
from transformers import pipeline

# Build a description from the top-scoring frames' best-matched prompts
@torch.no_grad()
def get_top_matched_prompts(frames_data, clip_scores_norm, highlight_prompts, top_n=5):
    """Find which prompts most describe the highlight frames."""
    top_frame_indices = np.argsort(clip_scores_norm)[-top_n:]
    top_images = [frames_data[i][2] for i in top_frame_indices]
    
    img_inputs = clip_processor(images=top_images, return_tensors='pt', padding=True).to(DEVICE)
    img_features = clip_model.get_image_features(**img_inputs)
    img_features = img_features / img_features.norm(dim=-1, keepdim=True)
    
    txt_inputs = clip_processor(text=highlight_prompts, return_tensors='pt', padding=True).to(DEVICE)
    txt_features = clip_model.get_text_features(**txt_inputs)
    txt_features = txt_features / txt_features.norm(dim=-1, keepdim=True)
    
    sim = (img_features @ txt_features.T).mean(dim=0).cpu().numpy()
    ranked = np.argsort(sim)[::-1]
    return [highlight_prompts[i] for i in ranked[:3]]

top_prompts = get_top_matched_prompts(frames_data, clip_scores_norm, HIGHLIGHT_PROMPTS)
print(f'Top matched highlight themes:')
for p in top_prompts:
    print(f'  → {p}')

# Summarize with BART
print('\n📥 Loading BART summarizer...')
summarizer = pipeline('summarization', model='facebook/bart-large-cnn', device=0 if DEVICE=='cuda' else -1)

description_text = (
    f"This highlight reel was automatically generated from a video using CLIP-based scene scoring. "
    f"The top highlight themes identified were: {', '.join(top_prompts)}. "
    f"A total of {len(highlight_segments)} key segments were selected spanning "
    f"{total_duration:.0f} seconds of the most visually significant moments in the video."
)

summary = summarizer(description_text, max_length=60, min_length=20, do_sample=False)
print(f'\n📝 Auto-generated highlight summary:')
print(f'   "{summary[0]["summary_text"]}"')

## Step 11 — Final Output Summary

In [ ]:
print('=' * 55)
print('  🎬 HIGHLIGHT GENERATOR — FINAL RESULTS')
print('=' * 55)
print(f'  Source video      : {video_path}')
print(f'  Source duration   : {duration:.1f}s ({duration/60:.1f} min)')
print(f'  Frames extracted  : {len(frames_data)}')
print(f'  Segments selected : {len(highlight_segments)}')
print(f'  Highlight length  : {total_duration:.1f}s ({total_duration/60:.1f} min)')
print(f'  Compression ratio : {(total_duration/duration)*100:.1f}% of original')
print(f'  Output file       : {OUTPUT_PATH}')
print(f'  Output size       : {os.path.getsize(OUTPUT_PATH)/1e6:.1f} MB')
print('=' * 55)
print('\n📁 Output files:')
print(f'  highlight_reel.mp4         ← Main output')
print(f'  clip_scores_timeline.png   ← Score visualization')
print(f'  sample_frames.png          ← Frame preview')

# Show score timeline again for reference
from IPython.display import Image as IPImage, display
display(IPImage('clip_scores_timeline.png'))

## Bonus — Evaluate Against TVSum Ground Truth

TVSum provides human-annotated importance scores per frame. We compare our CLIP scores against them.

In [ ]:
from scipy.stats import kendalltau, spearmanr

# Resample GT scores to match our extracted frame count
gt_resampled = np.interp(
    np.linspace(0, len(gt_scores)-1, len(clip_scores_norm)),
    np.arange(len(gt_scores)),
    gt_scores
)
gt_norm = (gt_resampled - gt_resampled.min()) / (gt_resampled.max() - gt_resampled.min() + 1e-8)

tau, p_tau = kendalltau(clip_scores_norm, gt_norm)
rho, p_rho = spearmanr(clip_scores_norm, gt_norm)

print('📊 Evaluation vs TVSum Ground Truth:')
print(f'  Kendall\'s Tau  : {tau:.4f}  (p={p_tau:.4f})')
print(f'  Spearman Rho   : {rho:.4f}  (p={p_rho:.4f})')

# Plot comparison
plt.figure(figsize=(16, 4))
plt.plot(timestamps, clip_scores_norm, label='CLIP Score (ours)', color='navy', linewidth=1.5)
plt.plot(timestamps, gt_norm, label='Human GT Score (TVSum)', color='darkorange', linewidth=1.5, alpha=0.8)
plt.xlabel('Time (seconds)')
plt.ylabel('Normalized Score')
plt.title(f'CLIP Score vs Human Ground Truth  |  Kendall τ={tau:.3f}  Spearman ρ={rho:.3f}', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('clip_vs_gt.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Evaluation chart saved.')

---

## Project Summary

| Component | Details |
|---|---|
| **Model** | CLIP ViT-B/32 (OpenAI, via HuggingFace) |
| **Summarizer** | BART-large-CNN (Meta, via HuggingFace) |
| **Dataset** | TVSum50 (Yale) |
| **Video download** | yt-dlp (free, no API key) |
| **Video stitching** | MoviePy |
| **Evaluation** | Kendall's Tau + Spearman vs TVSum GT |
| **Platform** | Kaggle P100 GPU |
| **Paid APIs** | None |

### How it works
1. Frames are extracted from the source video every 2 seconds
2. Each frame is encoded by CLIP into a 512-dim embedding
3. Highlight prompts are also encoded by CLIP
4. Cosine similarity between frame and prompt embeddings gives a highlight score
5. High-scoring frames are grouped into continuous segments
6. Segments are stitched into the final highlight reel
7. Evaluation compares our scores against human-annotated TVSum ground truth

### Resume line
> *Built an Automatic Video Highlight Generator using CLIP ViT-B/32 for multimodal scene scoring and temporal segmentation, evaluated against TVSum50 human annotations. Deployed on Kaggle P100 GPU with zero paid APIs.*